In [1]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, round as sround

df = spark.table("gold_loan_master").na.fill(0)

# Categorical String Indexing
idx_emp = StringIndexer(inputCol="employment_type", outputCol="emp_idx", handleInvalid="keep")
idx_ltv = StringIndexer(inputCol="ltv_band", outputCol="ltv_idx", handleInvalid="keep")
idx_cibil = StringIndexer(inputCol="cibil_band", outputCol="cibil_idx", handleInvalid="keep")

StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 3, Finished, Available, Finished, False)

In [2]:
# Feature Vector Assembly
feature_cols = [
    "disbursed_amount", "asset_cost", "ltv", "cibil_score", "kyc_count",
    "pri_overdue_accts", "total_overdue_accts", "total_current_balance",
    "pri_instal_amt", "new_accts_6m", "delinquent_accts_6m",
    "avg_acct_age_months", "credit_history_months", "no_of_inquiries",
    "overdue_ratio", "age_years", "emp_idx", "ltv_idx", "cibil_idx"
]

StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 4, Finished, Available, Finished, False)

In [3]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")
rf = RandomForestClassifier(labelCol="is_default", featuresCol="features", numTrees=100, maxDepth=10, seed=42)

pipeline = Pipeline(stages=[idx_emp, idx_ltv, idx_cibil, assembler, rf])


StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 5, Finished, Available, Finished, False)

In [4]:

# Train / Test Split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_df)

StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 6, Finished, Available, Finished, False)

In [5]:
# Evaluate on Test Set
test_predictions = model.transform(test_df)
evaluator = BinaryClassificationEvaluator(labelCol="is_default", metricName="areaUnderROC")
auc_score = evaluator.evaluate(test_predictions)
print(f"📊 LTFS Model Test Set AUC-ROC: {round(auc_score, 4)}")

StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 7, Finished, Available, Finished, False)

📊 LTFS Model Test Set AUC-ROC: 0.6307


In [6]:
# Score the Full Portfolio
full_predictions = model.transform(df)

scored_df = (
    full_predictions
    .withColumn("prob_array", vector_to_array("probability"))
    .withColumn("default_probability_pct", sround(col("prob_array")[1] * 100, 2))
    .withColumn("npa_risk_tier",
        when(col("default_probability_pct") >= 65.0, "High Risk")
        .when(col("default_probability_pct") >= 35.0, "Medium Risk")
        .otherwise("Low Risk")
    )
    .select(
        "loan_id", "branch_id", "state_id", "disbursal_date",
        "disbursed_amount", "ltv", "ltv_band",
        "cibil_score", "cibil_band", "employment_type",
        "is_default", "default_status_label",
        "prediction", "default_probability_pct", "npa_risk_tier"
    )
)

scored_df.write.mode("overwrite").format("delta").saveAsTable("gold_loan_scores")
print("✅ Final Scored Predictions saved to Delta table: gold_loan_scores")

StatementMeta(, 09589285-8b2d-47c1-b36d-ddc82c037600, 8, Finished, Available, Finished, False)

✅ Final Scored Predictions saved to Delta table: gold_loan_scores
